In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
TRAIN_IMG = "/content/drive/MyDrive/Dataset/train/img"
TRAIN_ANN = "/content/drive/MyDrive/Dataset/train/ann"

VAL_IMG = "/content/drive/MyDrive/Dataset/valid/img"
VAL_ANN = "/content/drive/MyDrive/Dataset/valid/ann"

In [4]:
!pip install tensorflow opencv-python scikit-learn matplotlib

In [9]:
import os
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [6]:
def load_data(img_dir, ann_dir):
    images = []
    labels = []

    for ann_file in os.listdir(ann_dir):
        if ann_file.endswith(".json"):
            with open(os.path.join(ann_dir, ann_file)) as f:
                data = json.load(f)

            img_path = os.path.join(img_dir, ann_file.replace(".json", ""))
            img = cv2.imread(img_path)

            if img is None:
                continue

            img = cv2.resize(img, (224, 224))

            for obj in data.get("objects", []):
                label = obj.get("label") or obj.get("name") or obj.get("classTitle")
                if label:
                    images.append(img)
                    labels.append(label)

    return np.array(images), np.array(labels)

In [8]:
X_train, y_train = load_data(TRAIN_IMG, TRAIN_ANN)
X_val, y_val = load_data(VAL_IMG, VAL_ANN)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))

Training samples: 21764
Validation samples: 2584


In [10]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_val = encoder.transform(y_val)

print("Classes:", encoder.classes_)

Classes: ['ambulance' 'army vehicle' 'auto rickshaw' 'bicycle' 'bus' 'car'
 'garbagevan' 'human hauler' 'minibus' 'minivan' 'motorbike' 'pickup'
 'policecar' 'rickshaw' 'scooter' 'suv' 'taxi' 'three wheelers -CNG-'
 'truck' 'van' 'wheelbarrow']


In [1]:
X_train = X_train / 255.0
X_val = X_val / 255.0

NameError: name 'X_train' is not defined

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    MaxPooling2D(),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(encoder.classes_), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=16
)

In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title("Loss")
plt.legend()

plt.show()

In [ ]:
pred = model.predict(X_val)
pred_classes = encoder.inverse_transform(np.argmax(pred, axis=1))

print("Sample Predictions:")
for i in range(5):
    print("Predicted:", pred_classes[i])

In [ ]:
def get_priority(vehicle):
    priority = {
        "ambulance": "HIGH PRIORITY 🚑",
        "fire_truck": "HIGH PRIORITY 🚒",
        "police": "MEDIUM PRIORITY 🚓",
        "car": "NORMAL",
        "bus": "NORMAL"
    }
    return priority.get(vehicle.lower(), "NORMAL")

for v in pred_classes[:5]:
    print(v, "→", get_priority(v))